# 03. Data Preprocessing - Data Cleaning Pipeline
Clean raw data: fix numbers stored as text, fix the employment placeholder, and drop the ID column.

## 1. Load Data
Import cleaning functions and load the dataset.

In [1]:
import os
import sys
import pandas as pd
import numpy as np

# Add project root to path
sys.path.append(os.path.abspath('..'))

from src.data.data_cleaning import (
    clean_corrupted_numerical_columns,
    handle_employed_sentinel,
    remove_identifiers,
    clean_raw_data
)

# Load raw dataset
data_path = '../data/raw/Train_Dataset.csv' if os.path.exists('../data/raw/Train_Dataset.csv') else 'data/raw/Train_Dataset.csv'
df = pd.read_csv(data_path, low_memory=False)

print(f"Loaded raw data: {df.shape[0]:,} rows and {df.shape[1]} columns.")


Loaded raw data: 121,856 rows and 40 columns.


## 2. Fix Numbers Stored as Text
Convert columns with symbols like `$`, `x`, `&` into clean numbers.

In [2]:
target_cols = ['Client_Income', 'Credit_Amount', 'Loan_Annuity', 'Age_Days', 'Employed_Days', 'Score_Source_3']

# Check data types before cleaning
print("Data types before cleaning:")
print(df[target_cols].dtypes)

# Apply cleaning
df = clean_corrupted_numerical_columns(df)

# Check data types after cleaning
print("\nData types after cleaning:")
print(df[target_cols].dtypes)


Data types before cleaning:
Client_Income     str
Credit_Amount     str
Loan_Annuity      str
Age_Days          str
Employed_Days     str
Score_Source_3    str
dtype: object

Data types after cleaning:
Client_Income     float64
Credit_Amount     float64
Loan_Annuity      float64
Age_Days          float64
Employed_Days     float64
Score_Source_3    float64
dtype: object


### Takeaways:
- Text numbers are now clean `float` numbers.
- Symbols (`$`, `x`, `&`) were converted to `NaN`.

## 3. Fix Employment Days (365243)
The value 365,243 means about 1,000 years. It is a placeholder for retired or unemployed people.
We flag them in `Is_Retired_Unemployed` and set their employment days to `NaN`.

In [3]:
# Apply sentinel handling
df = handle_employed_sentinel(df)

# Check the new binary flag
print("Counts for 'Is_Retired_Unemployed':")
print(df['Is_Retired_Unemployed'].value_counts())

# Verify 365243 is removed from Employed_Days
print("\nCount of 365243 in Employed_Days:", (df['Employed_Days'] == 365243).sum())
print("Realistic max Employed_Days:", df['Employed_Days'].max(), "days (~48 years)")


Counts for 'Is_Retired_Unemployed':
Is_Retired_Unemployed
0    100758
1     21098
Name: count, dtype: int64

Count of 365243 in Employed_Days: 0
Realistic max Employed_Days: 17546.0 days (~48 years)


### Takeaways:
- Added `Is_Retired_Unemployed` column for retired and unemployed applicants.
- Removed 365,243 so employment days are realistic.

## 4. Drop ID Column
The `ID` column has no predictive value and can cause overfitting, so we drop it.

In [4]:
# Remove ID column
df = remove_identifiers(df)

print("Is 'ID' column still present?", 'ID' in df.columns)
print("Updated dataset shape:", df.shape)


Is 'ID' column still present? False
Updated dataset shape: (121856, 40)


### Takeaways:
- `ID` column is dropped.

## 5. Clean All in One Step
Test the `clean_raw_data()` function to run all steps together.

In [5]:
# Test the all-in-one cleaning function
raw_data = pd.read_csv(data_path, low_memory=False)
cleaned_df = clean_raw_data(raw_data)

print("Full cleaning pipeline complete!")
print("Cleaned data shape:", cleaned_df.shape)
print("'ID' present:", 'ID' in cleaned_df.columns)
print("'Is_Retired_Unemployed' present:", 'Is_Retired_Unemployed' in cleaned_df.columns)


Full cleaning pipeline complete!
Cleaned data shape: (121856, 40)
'ID' present: False
'Is_Retired_Unemployed' present: True


### Next Steps:
Next, we will handle missing values, outliers, and categorical encoding.